In [40]:
import openai
import json
import requests

openai_client = openai.OpenAI()
api_client = requests.Session() 
messages = []

In [41]:

def get_popular_movies():
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies"  
    response = api_client.get(url)  
    
    movies = response.json()
    sorted_movies = sorted(movies, key=lambda x: x['popularity'], reverse=True)
    top_movies = sorted_movies[:3]
    final_results = []
    for movie in top_movies:
        final_results.append({
            "id": movie["id"],
            "title": movie["title"],
            "overview": movie["overview"],
            "popularity": movie["popularity"] 
        })

    return json.dumps(final_results)



def get_movie_details(id):                      
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}"  
    response = api_client.get(url)
    return response.json()  

def get_similar_movies(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/similar" 
    response = api_client.get(url)
    return response.json()  

FUNCTION_MAP = {
    'get_popular_movies' : get_popular_movies,
    'get_movie_details' : get_movie_details,
    'get_similar_movies' : get_similar_movies
}



TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "function to get popular movie list.",
            "parameters": {
                "type": "object",
                "properties": {} ,
            "required": [] 
            }
        }
    }
    ,
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "function to get details of a specific movie.",
            "parameters": {
                "type": "object",
                "properties": {  
                    "id": {
                        "type": "string",  
                        "description": "ID of a movie"  
                    }
                },
                "required": ["id"] 
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "function to get similar movies with a specific movie.",
            "parameters": {
                "type": "object",
                "properties": {  
                    "id": {
                        "type": "string",
                        "description": "ID of a movie"
                    }
                },
                "required": ["id"] 
            }
        }
    }
]






In [42]:
from openai.types.chat import ChatCompletionMessage


def call_ai():
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function" : {
                        "name" : tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
                for tool_call in message.tool_calls

                ],
            }
        )
        
        for tool_call in message.tool_calls:

            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")    
        
            try:
                arguments = json.loads(arguments)   
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)   

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result),
                }
            )
        call_ai()
    else:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
        })
        print(f"AI: {message.content}")


initial_message = "Hi! I'm your Movie AI. Ask about movies: top lists, details, or personalized recommendations!"

print(f"AI: {initial_message}")
messages.append({"role": "assistant", "content": initial_message})

while True:
    user_input = input("Send a message to the LLM...")
    if user_input == "quit" or user_input == "q"  :
        break
    else:
        messages.append({"role": "user", "content":user_input})
        print(f"User: {user_input}")
        call_ai()

AI: Hi! I'm your Movie AI. Ask about movies: top lists, details, or personalized recommendations!
User: Let me know populat movies these day
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{"id": 1290821, "title": "Shelter", "overview": "A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.", "popularity": 506.2271}, {"id": 799882, "title": "The Bluff", "overview": "When her tranquil life on a remote island is shattered by the return of her vengeful former captain, a skilled ex-pirate must confront her bloody past and unleash her deadly talents to save her family from a ruthless siege.", "popularity": 346.9326}, {"id": 1236153, "title": "Mercy", "overview": "In the near future, a detective stands on trial accused of murdering his wife. He has ninety minutes to prove his innocence to the 

In [43]:
messages


[{'role': 'assistant',
  'content': "Hi! I'm your Movie AI. Ask about movies: top lists, details, or personalized recommendations!"},
 {'role': 'user', 'content': 'Let me know populat movies these day'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_CrCPt5qmk1a6NDo1fkm4FFyX',
    'type': 'function',
    'function': {'name': 'get_popular_movies', 'arguments': '{}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_CrCPt5qmk1a6NDo1fkm4FFyX',
  'name': 'get_popular_movies',
  'content': '"[{\\"id\\": 1290821, \\"title\\": \\"Shelter\\", \\"overview\\": \\"A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.\\", \\"popularity\\": 506.2271}, {\\"id\\": 799882, \\"title\\": \\"The Bluff\\", \\"overview\\": \\"When her tranquil life on a remote island is shattered by the return of her vengeful former captain, a skilled ex-